# Q2 - Lexical Candidate Generation (BM25)

Builds independent from-scratch BM25 indexes (one per dataset, via
`cs4406m26_assignment1c1.bm25`) over the unified `articles` feature store from
Q1, retrieves top-K candidates per user from their pre-window click history,
and reports recall@K per SPEC.md's Q2 section.

Run top-to-bottom (or via `python bm25_retrieval.py`) to rebuild
`data/processed/{dataset}/bm25_topk.parquet` and `bm25_metrics.json`.

Loading/filtering uses `polars` (not `pandas`) throughout, same reasoning as
`src/build_pipeline.ipynb`: at `ebnerd_large` scale (24.6M behavior rows),
polars' native list dtype and column projection keep memory bounded where
pandas' object-dtype list columns would not. A `BUILD_LARGE_ONLY` flag (same
convention as Q1) controls whether this run recomputes only `ebnerd_large`/
`mind_large` or all five tracks; progress is appended to `build_progress.log`
at the repo root throughout.

In [1]:
from datetime import datetime, timezone
from pathlib import Path
import gc
import json
import shutil

import numpy as np
import polars as pl

from cs4406m26_assignment1c1.bm25 import tokenize, build_index, top_k


def find_repo_root(marker: str = "pyproject.toml") -> Path:
    for parent in [Path.cwd(), *Path.cwd().parents]:
        if (parent / marker).exists():
            return parent
    raise FileNotFoundError(f"could not locate {marker} above {Path.cwd()}")


ROOT = find_repo_root()
DATA_DIR = ROOT / "data" / "processed"
PROGRESS_LOG = ROOT / "build_progress.log"

# Same flag/convention as src/build_pipeline.ipynb: when True, only the two
# large-scale tracks are (re)computed this run. ebnerd/ebnerd_small/mind are
# never read into feature_store nor written to below, so their already-
# persisted bm25_topk.parquet/bm25_metrics.json are left untouched.
BUILD_LARGE_ONLY = True
DATASETS = (
    ["ebnerd_large", "mind_large"]
    if BUILD_LARGE_ONLY
    else ["ebnerd", "ebnerd_small", "mind", "ebnerd_large", "mind_large"]
)


def log_progress(message: str) -> None:
    # Appends only -- unlike build_pipeline.ipynb's PROGRESS_LOG (truncated
    # once at the very start of the Q1-Q5 pipeline), every downstream
    # notebook appends so build_progress.log stays one continuous log.
    with PROGRESS_LOG.open("a", encoding="utf-8") as f:
        f.write(f"[{datetime.now(timezone.utc).isoformat()}] {message}\n")
        f.flush()


RECENT_N_CLICKS = 20
BM25_K1 = 1.5
BM25_B = 0.75
CANDIDATE_K_VALUES = [50, 100, 200]
TOPK_MAX = max(CANDIDATE_K_VALUES)

log_progress(f"bm25_retrieval started (BUILD_LARGE_ONLY={BUILD_LARGE_ONLY}, datasets={DATASETS})")

feature_store = {}
for name in DATASETS:
    feature_store[name] = {
        "articles": pl.read_parquet(DATA_DIR / name / "articles.parquet", columns=["article_id", "title", "abstract"]),
        "behaviors": pl.read_parquet(DATA_DIR / name / "behaviors.parquet", columns=["user_id", "article_ids_clicked", "split"]),
        "history": pl.read_parquet(DATA_DIR / name / "history.parquet", columns=["user_id", "article_id_sequence"]),
    }
    log_progress(
        f"  {name}: loaded articles/behaviors/history "
        f"({feature_store[name]['articles'].height}, {feature_store[name]['behaviors'].height}, "
        f"{feature_store[name]['history'].height} rows)"
    )

{name: {table: df.shape for table, df in tables.items()} for name, tables in feature_store.items()}

{'ebnerd_large': {'articles': (125541, 3),
  'behaviors': (24630275, 3),
  'history': (974791, 2)},
 'mind_large': {'articles': (104151, 3),
  'behaviors': (2609219, 3),
  'history': (750434, 2)}}

## Tokenizer

`tokenize` is imported from `cs4406m26_assignment1c1.bm25` (lowercase +
Unicode-aware `\w+` splitting - works for both Danish and English without a
new dependency, since Python's `re` treats `\w` as Unicode by default, so
`æøå`/`ÆØÅ` stay intact as word characters). This is a smoke test of the
imported function, not a redefinition.

In [2]:
def test_tokenize():
    assert tokenize(None) == []
    assert tokenize("") == []
    assert tokenize("ÆØÅ æble Rødgrød") == ["æøå", "æble", "rødgrød"]
    assert tokenize("Harry's dna-test") == ["harry", "s", "dna", "test"]


test_tokenize()
print("ok: tokenizer handles Danish/English text and null/empty input")

ok: tokenizer handles Danish/English text and null/empty input


## Per-dataset tokenized corpus (title + abstract)

`abstract` must be `.fillna("")`'d before concatenation - ~5.2% of MIND
articles have a null abstract, and naive concatenation would propagate NaN and
silently zero out those articles' tokens.

In [3]:
corpus = {}
for name in DATASETS:
    articles = feature_store[name]["articles"]
    texts = (articles["title"].fill_null("") + " " + articles["abstract"].fill_null("")).to_list()
    doc_tokens = [tokenize(t) for t in texts]
    corpus[name] = {
        "doc_ids": articles["article_id"].to_numpy(),
        "doc_tokens": doc_tokens,
        "doc_len": np.array([len(t) for t in doc_tokens]),
    }

{name: {"n_docs": len(c["doc_tokens"]), "avg_doc_len": c["doc_len"].mean()} for name, c in corpus.items()}

{'ebnerd_large': {'n_docs': 125541,
  'avg_doc_len': np.float64(24.37731896352586)},
 'mind_large': {'n_docs': 104151,
  'avg_doc_len': np.float64(48.56869353150714)}}

In [4]:
def test_corpus_alignment():
    for name in DATASETS:
        articles = feature_store[name]["articles"]
        c = corpus[name]
        assert len(c["doc_tokens"]) == len(articles)
        assert (c["doc_ids"] == articles["article_id"].to_numpy()).all()

    # At least one dataset in scope should have null abstracts (MIND-family
    # datasets do, ~5%) -- must not silently zero out those articles' tokens.
    # Not hardcoded to "mind" so this still works when BUILD_LARGE_ONLY swaps
    # the dataset list to ["ebnerd_large", "mind_large"].
    null_abstract_dataset = next(
        (name for name in DATASETS if feature_store[name]["articles"]["abstract"].is_null().sum() > 0),
        None,
    )
    assert null_abstract_dataset is not None, "expected at least one dataset with null abstracts in scope"
    articles = feature_store[null_abstract_dataset]["articles"]
    null_mask = articles["abstract"].is_null().to_numpy()
    sample_pos = np.flatnonzero(null_mask)[0]
    assert len(corpus[null_abstract_dataset]["doc_tokens"][sample_pos]) > 0, "null-abstract article lost its title tokens"


test_corpus_alignment()
print("ok: tokenized corpus aligned with articles and null-abstract rows still have tokens")

ok: tokenized corpus aligned with articles and null-abstract rows still have tokens


## Build a BM25Index per dataset

From-scratch postings-list inverted index (`cs4406m26_assignment1c1.bm25.build_index`):
`term -> (doc_idx array, tf array)`, plus `idf`, `doc_len`, `avgdl` computed
directly from the tokenized corpus - no third-party BM25 library. See SPEC.md
Q2 section 1 for why this is hand-built rather than a library call
(`rank_bm25`'s own scoring measured at ~2.9s/query on MIND vs. ~5ms/query
here - a library was evaluated and rejected on performance grounds).

In [5]:
bm25_index = {
    name: build_index(corpus[name]["doc_ids"], corpus[name]["doc_tokens"], k1=BM25_K1, b=BM25_B)
    for name in DATASETS
}

for name in DATASETS:
    idx = bm25_index[name]
    print(f"{name}: {idx.n_docs} docs, avgdl={idx.avgdl:.2f}, vocab={len(idx.idf)} terms")

ebnerd_large: 125541 docs, avgdl=24.38, vocab=120531 terms
mind_large: 104151 docs, avgdl=48.57, vocab=75091 terms


In [6]:
def test_bm25_index_matches_corpus():
    for name in DATASETS:
        idx = bm25_index[name]
        c = corpus[name]
        assert idx.n_docs == len(c["doc_tokens"])
        assert np.isclose(idx.avgdl, c["doc_len"].mean())
        assert np.array_equal(idx.doc_len, c["doc_len"])
        assert all(v >= 0 for v in idx.idf.values()), "non-negative IDF variant must hold for every term"

        # by-hand IDF check for a couple of real terms
        n_docs = idx.n_docs
        for term, (doc_idx, _tf) in list(idx.postings.items())[:5]:
            df = len(doc_idx)
            expected_idf = np.log((n_docs - df + 0.5) / (df + 0.5) + 1.0)
            assert np.isclose(idx.idf[term], expected_idf)


test_bm25_index_matches_corpus()
print("ok: BM25Index (doc_len/avgdl/idf/postings) matches our own tokenized corpus")

ok: BM25Index (doc_len/avgdl/idf/postings) matches our own tokenized corpus


## Top-K retrieval

Candidate generation retrieves from the whole article catalog (not a
re-ranking of the impression's given `article_ids_inview`).

In [7]:
def top_k_candidates(query_tokens: list[str], dataset: str, k: int = TOPK_MAX) -> list[tuple[str, float]]:
    return top_k(bm25_index[dataset], query_tokens, k)

In [8]:
def test_top_k_candidates():
    for name in DATASETS:
        sample_title = feature_store[name]["articles"]["title"][0]
        q = tokenize(sample_title)
        top200 = top_k_candidates(q, name, k=200)
        top50 = top_k_candidates(q, name, k=50)
        assert len(top200) == 200
        assert len(top50) == 50
        scores200 = [s for _, s in top200]
        assert scores200 == sorted(scores200, reverse=True)
        assert top200[:50] == top50, "top-50 must be a prefix of top-200 (nesting invariant)"
        assert top_k_candidates([], name, k=50) == []


test_top_k_candidates()
print("ok: top-K retrieval is sorted, nested across K, and empty for an empty query")

ok: top-K retrieval is sorted, nested across K, and empty for an empty query


## Query construction from history

Concatenates the titles of a user's most recent `RECENT_N_CLICKS` clicks.
`timestamp_sequence` is chronologically ascending, so "most recent N" is the
*last* N elements of `article_id_sequence`, not the first N.

In [9]:
def build_user_query_tokens(article_id_sequence, title_lookup: dict, recent_n: int = RECENT_N_CLICKS) -> list[str]:
    recent_ids = list(article_id_sequence)[-recent_n:]
    titles = [title_lookup.get(aid, "") for aid in recent_ids]
    return tokenize(" ".join(t for t in titles if t))


title_by_id = {
    name: dict(zip(feature_store[name]["articles"]["article_id"].to_list(), feature_store[name]["articles"]["title"].to_list()))
    for name in DATASETS
}

In [10]:
def test_build_user_query_tokens():
    assert build_user_query_tokens([], {}) == []

    lookup = {"a": "OLDWORD", "b": "OLDWORD2", "c": "RECENTWORD"}
    ids = ["a", "b", "c"]
    assert build_user_query_tokens(ids, lookup, recent_n=1) == ["recentword"]
    tokens_all = build_user_query_tokens(ids, lookup, recent_n=10)
    assert "oldword" in tokens_all and "recentword" in tokens_all


test_build_user_query_tokens()
print("ok: query construction takes the most recent N clicks (last N, not first N)")

ok: query construction takes the most recent N clicks (last N, not first N)


## Per-user retrieval cache (val/test users only)

The query depends only on the user's fixed pre-window `history`, identical for
every impression of that user - so retrieval only needs to run once per user,
not once per impression. Only users appearing in `val`/`test` need retrieval
computed (recall@K isn't reported on `train`).

In [11]:
user_topk = {}
coldstart_users = {}
for name in DATASETS:
    behaviors = feature_store[name]["behaviors"]
    history = feature_store[name]["history"]
    eval_user_ids = set(behaviors.filter(pl.col("split").is_in(["val", "test"]))["user_id"].to_list())
    history_eval = history.filter(pl.col("user_id").is_in(list(eval_user_ids)))
    lookup = title_by_id[name]

    topk_for_dataset = {}
    coldstart = set()
    user_ids_col = history_eval["user_id"].to_list()
    sequences_col = history_eval["article_id_sequence"].to_list()
    n_users = len(user_ids_col)
    log_progress(f"  {name}: retrieving BM25 top-K for {n_users} eval users")
    for i, (user_id, article_id_sequence) in enumerate(zip(user_ids_col, sequences_col)):
        query_tokens = build_user_query_tokens(article_id_sequence, lookup)
        if not query_tokens:
            coldstart.add(user_id)
            continue
        topk_for_dataset[user_id] = top_k_candidates(query_tokens, name, k=TOPK_MAX)
        if (i + 1) % 50_000 == 0:
            log_progress(f"    {name}: {i + 1}/{n_users} users retrieved")

    user_topk[name] = topk_for_dataset
    coldstart_users[name] = coldstart
    log_progress(f"  {name}: retrieval done -- {len(topk_for_dataset)} retrieved, {len(coldstart)} cold-start")

{name: {"retrieved": len(user_topk[name]), "coldstart": len(coldstart_users[name])} for name in DATASETS}

{'ebnerd_large': {'retrieved': 821111, 'coldstart': 0},
 'mind_large': {'retrieved': 415122, 'coldstart': 10423}}

In [12]:
def test_user_retrieval_cache():
    for name in DATASETS:
        behaviors = feature_store[name]["behaviors"]
        eval_user_ids = set(behaviors.filter(pl.col("split").is_in(["val", "test"]))["user_id"].to_list())
        assert set(user_topk[name]).issubset(eval_user_ids)
        assert set(coldstart_users[name]).issubset(eval_user_ids)
        assert set(user_topk[name]) | set(coldstart_users[name]) == eval_user_ids

        history = feature_store[name]["history"]
        history_eval = history.filter(pl.col("user_id").is_in(list(eval_user_ids)))
        expected_coldstart = set(
            history_eval.filter(pl.col("article_id_sequence").list.len() == 0)["user_id"].to_list()
        )
        assert coldstart_users[name] == expected_coldstart, f"{name}: cold-start set mismatch"

        for aid_scores in list(user_topk[name].values())[:5]:
            assert 1 <= len(aid_scores) <= TOPK_MAX


test_user_retrieval_cache()
print("ok: retrieval cache covers exactly the val/test user population, cold-start counted independently")

ok: retrieval cache covers exactly the val/test user population, cold-start counted independently


## Recall@K evaluation

Fractional multi-relevant definition (`article_ids_clicked` is not always
singleton), macro-averaged over non-cold-start impressions per split.

In [13]:
def evaluate_recall(dataset: str, split: str, k_values: list[int]) -> dict:
    behaviors = feature_store[dataset]["behaviors"]
    split_behaviors = behaviors.filter(pl.col("split") == split)
    topk = user_topk[dataset]
    coldstart = coldstart_users[dataset]

    recalls = {k: [] for k in k_values}
    n_excluded = 0

    for user_id, clicked in zip(split_behaviors["user_id"].to_list(), split_behaviors["article_ids_clicked"].to_list()):
        if user_id in coldstart:
            n_excluded += 1
            continue
        clicked = set(clicked)
        candidate_ids = [aid for aid, _ in topk[user_id]]
        for k in k_values:
            top_k_ids = set(candidate_ids[:k])
            recalls[k].append(len(clicked & top_k_ids) / len(clicked))

    n_evaluated = len(split_behaviors) - n_excluded
    return {
        "recall_at_k": {k: (sum(v) / len(v) if v else 0.0) for k, v in recalls.items()},
        "n_total": len(split_behaviors),
        "n_evaluated": n_evaluated,
        "n_excluded_coldstart": n_excluded,
    }


metrics = {name: {split: evaluate_recall(name, split, CANDIDATE_K_VALUES) for split in ["val", "test"]} for name in DATASETS}
log_progress(f"bm25_retrieval: recall@K computed for {DATASETS}")
metrics

{'ebnerd_large': {'val': {'recall_at_k': {50: 0.0012877293021772826,
    100: 0.0025979423728604936,
    200: 0.005289289878498027},
   'n_total': 1678989,
   'n_evaluated': 1678989,
   'n_excluded_coldstart': 0},
  'test': {'recall_at_k': {50: 0.002164121860985756,
    100: 0.0037251795860954342,
    200: 0.006342239317969808},
   'n_total': 12566385,
   'n_evaluated': 12566385,
   'n_excluded_coldstart': 0}},
 'mind_large': {'val': {'recall_at_k': {50: 0.010256034753601964,
    100: 0.01923655272619149,
    200: 0.02966792898206836},
   'n_total': 431517,
   'n_evaluated': 420124,
   'n_excluded_coldstart': 11393},
  'test': {'recall_at_k': {50: 0.003767289452891333,
    100: 0.00764219603795652,
    200: 0.013200163466336784},
   'n_total': 376471,
   'n_evaluated': 365201,
   'n_excluded_coldstart': 11270}}}

In [14]:
def test_recall_at_k_monotonic():
    for name in DATASETS:
        for split in ["val", "test"]:
            m = metrics[name][split]
            r = m["recall_at_k"]
            assert r[50] <= r[100] + 1e-12 <= r[200] + 1e-12
            assert all(0.0 <= v <= 1.0 for v in r.values())
            assert m["n_evaluated"] + m["n_excluded_coldstart"] == m["n_total"]

    # verified separately that neither dataset has zero-click impressions
    for name in DATASETS:
        b = feature_store[name]["behaviors"]
        assert (b["article_ids_clicked"].list.len() == 0).sum() == 0


test_recall_at_k_monotonic()
print("ok: recall@K is monotonic in K, bounded in [0,1], and impression counts are consistent")

ok: recall@K is monotonic in K, bounded in [0,1], and impression counts are consistent


## Persist BM25 outputs

Keyed by user (not impression) to avoid storing duplicate retrieval results
across a user's many impressions - mirrors Q1's `manifest.json` conventions.

In [ ]:
import pyarrow.parquet as pq


def write_topk_parquet_chunked(user_ids: list, topk_lists: list, out_path: Path, dataset: str, chunk_rows: int = 50_000) -> None:
    """Same chunked-write pattern as build_pipeline.ipynb's write_parquet_chunked
    (see SPEC.md Q1 #5, Q2 #10) -- building the whole pl.DataFrame in one shot
    OOM'd here at ebnerd_large scale (821,111 rows x two 200-element list
    columns, ~2.6GB single allocation) even though this path never touches
    pandas at all, since polars itself still has to materialize one large
    contiguous buffer per column. Each chunk is written as its own small
    parquet file. The merge step uses pyarrow's ParquetWriter directly
    (append one row group per chunk file) rather than
    polars.scan_parquet(...).sink_parquet(...): the latter OOM'd too, merging
    just 17 chunk files, because this polars version's sink_parquet still
    collects the full result into memory before writing rather than
    truly streaming -- pyarrow's row-group-append never holds more than one
    chunk's worth of data in memory regardless of the final file's total size.

    No explicit gc.collect() here: each chunk_df/table is a plain, non-cyclic
    object, so `del` alone frees it immediately via refcounting. A forced
    gc.collect() scans the *entire* live object graph for cycles -- with this
    notebook's large persistent state (BM25 indexes, the full user_topk
    dict), that full-heap scan cost minutes per call, not the near-zero cost
    it has on a small heap -- confirmed directly by comparing against
    embedding_retrieval.ipynb's equivalent write step, which took ~2+ hours
    with gc.collect() calls in this same loop and only minutes once removed."""
    n = len(user_ids)
    if n <= chunk_rows:
        pl.DataFrame({
            "user_id": user_ids,
            "dataset": [dataset] * n,
            "n_retrieved": [len(x) for x in topk_lists],
            "retrieved_article_ids": [[aid for aid, _ in x] for x in topk_lists],
            "retrieved_scores": [[float(s) for _, s in x] for x in topk_lists],
        }).write_parquet(out_path)
        return

    tmp_dir = out_path.parent / f"_{out_path.stem}_chunks_tmp"
    if tmp_dir.exists():
        shutil.rmtree(tmp_dir)
    tmp_dir.mkdir(parents=True)
    n_chunks = (n + chunk_rows - 1) // chunk_rows
    try:
        for i, start in enumerate(range(0, n, chunk_rows)):
            end = min(start + chunk_rows, n)
            chunk_topk = topk_lists[start:end]
            chunk_df = pl.DataFrame({
                "user_id": user_ids[start:end],
                "dataset": [dataset] * (end - start),
                "n_retrieved": [len(x) for x in chunk_topk],
                "retrieved_article_ids": [[aid for aid, _ in x] for x in chunk_topk],
                "retrieved_scores": [[float(s) for _, s in x] for x in chunk_topk],
            })
            chunk_df.write_parquet(tmp_dir / f"part_{i:04d}.parquet")
            del chunk_df
            log_progress(f"    {out_path.name}: chunk {i + 1}/{n_chunks} written")

        writer = None
        try:
            for chunk_path in sorted(tmp_dir.glob("part_*.parquet")):
                table = pq.read_table(chunk_path)
                if writer is None:
                    writer = pq.ParquetWriter(out_path, table.schema)
                writer.write_table(table)
                del table
        finally:
            if writer is not None:
                writer.close()
        log_progress(f"    {out_path.name}: merged {n_chunks} chunks")
    finally:
        shutil.rmtree(tmp_dir)


def write_bm25_outputs(dataset: str) -> Path:
    out_dir = DATA_DIR / dataset
    user_ids = list(user_topk[dataset].keys())
    topk_lists = list(user_topk[dataset].values())
    write_topk_parquet_chunked(user_ids, topk_lists, out_dir / "bm25_topk.parquet", dataset)
    log_progress(f"  {dataset}: wrote bm25_topk.parquet ({len(user_ids)} rows)")

    bm25_metrics = {
        "schema_version": 1,
        "build_timestamp": datetime.now(timezone.utc).isoformat(),
        "hyperparameters": {
            "k1": BM25_K1, "b": BM25_B,
            "recent_n_clicks": RECENT_N_CLICKS, "topk_max": TOPK_MAX,
        },
        "recall_at_k": {split: metrics[dataset][split]["recall_at_k"] for split in ["val", "test"]},
        "n_impressions": {
            split: {
                "total": metrics[dataset][split]["n_total"],
                "evaluated": metrics[dataset][split]["n_evaluated"],
                "excluded_coldstart": metrics[dataset][split]["n_excluded_coldstart"],
            }
            for split in ["val", "test"]
        },
        "scope": "val_test_users_only",
    }
    (out_dir / "bm25_metrics.json").write_text(json.dumps(bm25_metrics, indent=2))
    log_progress(f"  {dataset}: wrote bm25_metrics.json")
    return out_dir


bm25_out_dirs = {name: write_bm25_outputs(name) for name in DATASETS}
log_progress("bm25_retrieval: all outputs written")
bm25_out_dirs

In [16]:
def test_bm25_outputs_roundtrip():
    for name in DATASETS:
        out_dir = bm25_out_dirs[name]
        topk_path = out_dir / "bm25_topk.parquet"
        metrics_path = out_dir / "bm25_metrics.json"
        assert topk_path.exists() and metrics_path.exists()

        reloaded_topk = pl.read_parquet(topk_path)
        assert set(reloaded_topk["user_id"].to_list()) == set(user_topk[name])
        for row in reloaded_topk.head(20).iter_rows(named=True):
            assert len(row["retrieved_article_ids"]) == row["n_retrieved"]
            assert len(row["retrieved_scores"]) == row["n_retrieved"]

        reloaded_metrics = json.loads(metrics_path.read_text())
        for split in ["val", "test"]:
            for k in CANDIDATE_K_VALUES:
                expected = metrics[name][split]["recall_at_k"][k]
                actual = reloaded_metrics["recall_at_k"][split][str(k)]
                assert abs(expected - actual) < 1e-12


test_bm25_outputs_roundtrip()
log_progress("bm25_retrieval completed successfully")
print("ok: bm25_topk.parquet and bm25_metrics.json round-trip correctly for every dataset")

ok: bm25_topk.parquet and bm25_metrics.json round-trip correctly for every dataset
